# Ground Water Level Prediction Notebook

This notebook demonstrates data preparation, feature selection, model training, and evaluation for 1-month, 2-month, and 3-month ahead groundwater level forecasting.


## 1. Setup and Imports
Mount Google Drive and import necessary libraries.

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, LeakyReLU, Activation
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import matplotlib.pyplot as plt
!pip install HydroErr
import HydroErr as he
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from keras.models import load_model
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
# Set working directory
import os
os.chdir('/content/drive/My Drive/Shaki_bhai_aprl/Adapted_model_4.11.25/Output_2')

  Preparing metadata (setup.py) ... done
  Created wheel for HydroErr: filename=HydroErr-1.24-py3-none-any.whl size=20183 sha256=803dd6bc6c91cf63df7321005864784596e6184115832bca63139669e969ff26
  Stored in directory: /root/.cache/pip/wheels/30/c0/62/d4f417372149a7f442261b56c7dab62a70b5a4e6dd63b14510
Successfully built HydroErr
Mounted at /content/drive


## 2. Data Preparation
Load the CSV, parse dates, and engineer date features.

In [3]:
df = pd.read_csv('SS.001 from 1981-2017.csv')
df['Date Time'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date Time'].dt.month
df['Year'] = df['Date Time'].dt.year
df.set_index('Date Time', inplace=True)
df.drop(['Date', 'Day'], axis=1, errors='ignore', inplace=True)
df.head()

,yt,yt-1,yt-2,yt-3,yt-4,yt-5,yt-6,yt-7,yt-8,yt-9,...,CR-5,CR-6,CR-7,CR-8,CR-9,CR-10,CR-11,CR-12,Month,Year
Date Time,,,,,,,,,,,,,,,,,,,,,
1981-01-01,9.28,9.44,9.37,8.84,8.34,7.90,7.88,7.86,7.96,8.01,...,1033.0,1773.0,2033.5,2326.0,2444.0,2626.5,2654.5,2763.0,1,1981
1981-02-01,8.94,9.28,9.44,9.37,8.84,8.34,7.90,7.88,7.86,7.96,...,494.0,1063.0,1803.0,2063.5,2356.0,2474.0,2656.5,2684.5,2,1981
1981-03-01,8.77,8.94,9.28,9.44,9.37,8.84,8.34,7.90,7.88,7.86,...,326.5,575.5,1144.5,1884.5,2145.0,2437.5,2555.5,2738.0,3,1981
1981-04-01,8.61,8.77,8.94,9.28,9.44,9.37,8.84,8.34,7.90,7.88,...,290.5,456.5,705.5,1274.5,2014.5,2275.0,2567.5,2685.5,4,1981
1981-05-01,8.47,8.61,8.77,8.94,9.28,9.44,9.37,8.84,8.34,7.90,...,414.0,426.5,592.5,841.5,1410.5,2150.5,2411.0,2703.5,5,1981


## 3. Feature Selection
Compute mutual information and select top features.

In [4]:
mutual_info = mutual_info_regression(df, df['yt'])
mi_scores = pd.Series(mutual_info, index=df.columns).sort_values(ascending=False)
mi_scores.head(20)

,0
yt,4.443592
yt-1,1.190935
yt-2,0.597127
TmeanLag10,0.426045
CR-9,0.404513
yt-12,0.398617
yt-3,0.358829
TmeanLag5,0.352974
Month,0.347225
CR-8,0.344806


## 4. Prepare Feature Subsets
Select top-k features for different dataset sizes.

In [5]:
dataset_top15 = df.iloc[:, :15]
dataset_top10 = df.iloc[:, :10]
dataset_top8  = df.iloc[:, :8]
dataset_top6  = df.iloc[:, :6]
print(dataset_top15.shape, dataset_top10.shape, dataset_top8.shape, dataset_top6.shape)

(444, 15) (444, 10) (444, 8) (444, 6)


## 5. Model Definition
Define RMSE metric, callbacks, and model constructor.

In [6]:
def rmse(y_true, y_pred):
    return tf.sqrt(tf.reduce_mean(tf.square(y_pred - y_true)))
callbacks = [EarlyStopping(monitor='val_loss', patience=20, mode='min')]
def create_model(window_size, n_features, lstm_units=128, dense_units=64, dropout_rate=0.2):
    model = Sequential()
    model.add(LSTM(lstm_units, return_sequences=True, input_shape=(window_size, n_features)))
    model.add(LeakyReLU(alpha=0.5))
    model.add(LSTM(lstm_units, return_sequences=True))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(dropout_rate))
    model.add(LSTM(lstm_units//2))
    model.add(Dropout(dropout_rate))
    model.add(Dense(dense_units, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse', metrics=['mae', rmse])
    return model

## 6. Generate Time Series Data
Function to create sliding-window feature/target arrays.

In [7]:
# Define window size
window_size = 2
def generate_time_series_data(df, window_size, step):
    X_features, y_target = [], []
    for i in range(len(df) - window_size - step):
        X_features.append(df.iloc[i:i+window_size].values)
        y_target.append(df.iloc[i+window_size+step, 0])
    X = np.array(X_features)
    y = np.array(y_target).reshape(-1, 1)
    return X, y

## 7. Prepare Data for Forecast Horizons
Generate datasets for 1, 2, and 3 months ahead.

In [8]:
# 1-month ahead
step = 0
data_8, target_8 = generate_time_series_data(dataset_top8, window_size, step)
# 2-month ahead
step = 1
data_8_2, target_8_2 = generate_time_series_data(dataset_top8, window_size, step)
# 3-month ahead
step = 2
data_8_3, target_8_3 = generate_time_series_data(dataset_top8, window_size, step)

## 8. Scale and Split Data
MinMax scaling and train/test split for each horizon.

In [9]:
from sklearn.preprocessing import MinMaxScaler
# Scale and split function
def scale_and_split(X, y):
    scaler_X = MinMaxScaler()
    X_flat = X.reshape(-1, X.shape[-1])
    X_scaled = scaler_X.fit_transform(X_flat).reshape(X.shape)
    scaler_y = MinMaxScaler()
    y_scaled = scaler_y.fit_transform(y)
    split = -65
    return X_scaled[:split], X_scaled[split:], y_scaled[:split], y_scaled[split:], scaler_X, scaler_y
# Prepare for 1-month
X1_tr, X1_te, y1_tr, y1_te, sX1, sy1 = scale_and_split(data_8, target_8)
n_features = X1_tr.shape[2]

## 9. Train and Evaluate Models
Train separate models for each forecast horizon.

In [10]:
import time
def train_model(X_tr, y_tr, X_te, y_te, suffix):
    model = create_model(window_size, n_features)
    start = time.time()
    hist = model.fit(X_tr, y_tr, validation_split=0.2, epochs=50, batch_size=8, callbacks=callbacks)
    duration = time.time() - start
    loss, mae, rmse_val = model.evaluate(X_te, y_te)
    print(f"{suffix} ahead - time: {duration:.1f}s, loss: {loss:.4f}, mae: {mae:.4f}, rmse: {rmse_val:.4f}")
    return model, hist
# 1-month ahead
model1, hist1 = train_model(X1_tr, y1_tr, X1_te, y1_te, '1-month')
# 2-month ahead
X2_tr, X2_te, y2_tr, y2_te, sX2, sy2 = scale_and_split(data_8_2, target_8_2)
model2, hist2 = train_model(X2_tr, y2_tr, X2_te, y2_te, '2-month')
# 3-month ahead
X3_tr, X3_te, y3_tr, y3_te, sX3, sy3 = scale_and_split(data_8_3, target_8_3)
model3, hist3 = train_model(X3_tr, y3_tr, X3_te, y3_te, '3-month')

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - loss: 0.0859 - mae: 0.2277 - rmse: 0.2790 - val_loss: 0.0272 - val_mae: 0.1383 - val_rmse: 0.1544
Epoch 2/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0321 - mae: 0.1418 - rmse: 0.1731 - val_loss: 0.0234 - val_mae: 0.1369 - val_rmse: 0.1478
Epoch 3/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0176 - mae: 0.1073 - rmse: 0.1273 - val_loss: 0.0111 - val_mae: 0.0848 - val_rmse: 0.0944
Epoch 4/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0122 - mae: 0.0803 - rmse: 0.1018 - val_loss: 0.0116 - val_mae: 0.0914 - val_rmse: 0.1025
Epoch 5/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0102 - mae: 0.0761 - rmse: 0.0948 - val_loss: 0.0073 - val_mae: 0.0588 - val_rmse: 0.0733
Epoch 6/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0108 - mae: 0.0746 - rmse: 0.0964 - val_loss: 0.0069 - val_mae: 0.0622 - val_rmse: 0.0746
Epoch 7/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0092 - mae: 0.0662 - rmse: 0.0894

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


38/38 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - loss: 0.0957 - mae: 0.2385 - rmse: 0.2927 - val_loss: 0.0277 - val_mae: 0.1430 - val_rmse: 0.1572
Epoch 2/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0298 - mae: 0.1404 - rmse: 0.1679 - val_loss: 0.0302 - val_mae: 0.1494 - val_rmse: 0.1684
Epoch 3/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0286 - mae: 0.1324 - rmse: 0.1610 - val_loss: 0.0167 - val_mae: 0.1076 - val_rmse: 0.1216
Epoch 4/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0187 - mae: 0.1017 - rmse: 0.1283 - val_loss: 0.0114 - val_mae: 0.0807 - val_rmse: 0.0939
Epoch 5/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0179 - mae: 0.0996 - rmse: 0.1243 - val_loss: 0.0108 - val_mae: 0.0714 - val_rmse: 0.0886
Epoch 6/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0132 - mae: 0.0852 - rmse: 0.1082 - val_loss: 0.0148 - val_mae: 0.0834 - val_rmse: 0.1065
Epoch 7/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0164 - mae: 0.0931 - rmse: 0.1188 - val_loss

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 77ms/step - loss: 0.1128 - mae: 0.2762 - rmse: 0.3169 - val_loss: 0.0289 - val_mae: 0.1410 - val_rmse: 0.1556
Epoch 2/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.0357 - mae: 0.1519 - rmse: 0.1831 - val_loss: 0.0240 - val_mae: 0.1293 - val_rmse: 0.1449
Epoch 3/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0301 - mae: 0.1362 - rmse: 0.1653 - val_loss: 0.0164 - val_mae: 0.1005 - val_rmse: 0.1160
Epoch 4/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0300 - mae: 0.1269 - rmse: 0.1628 - val_loss: 0.0128 - val_mae: 0.0849 - val_rmse: 0.0964
Epoch 5/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0290 - mae: 0.1296 - rmse: 0.1573 - val_loss: 0.0139 - val_mae: 0.0903 - val_rmse: 0.1030
Epoch 6/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0181 - mae: 0.0983 - rmse: 0.1257 - val_loss: 0.0129 - val_mae: 0.0807 - val_rmse: 0.0924
Epoch 7/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0174 - mae: 0.0993 - rmse: 0.1264 - val_loss

## 10. Save Models and Results
Persist models and export predictions.

In [13]:
# Save models
model1.save('best_model_1.h5')
model2.save('best_model_2.h5')
model3.save('best_model_3.h5')


In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- 1‑month ahead predictions & save ---
y1_pred = model1.predict(X1_te)
y1_pred = sy1.inverse_transform(y1_pred)
y1_true = sy1.inverse_transform(y1_te)

results1 = pd.DataFrame({
    'Test Predictions': y1_pred.ravel(),
    'Actuals':          y1_true.ravel()
})
results1.to_csv('test_results_S001_1_month.csv', index=False)



# --- 2‑month ahead predictions & save ---
y2_pred = model2.predict(X2_te)
y2_pred = sy2.inverse_transform(y2_pred)
y2_true = sy2.inverse_transform(y2_te)

results2 = pd.DataFrame({
    'Test Predictions': y2_pred.ravel(),
    'Actuals':          y2_true.ravel()
})
results2.to_csv('test_results_S001_2_month.csv', index=False)


# --- 3‑month ahead predictions & save ---
y3_pred = model3.predict(X3_te)
y3_pred = sy3.inverse_transform(y3_pred)
y3_true = sy3.inverse_transform(y3_te)

results3 = pd.DataFrame({
    'Test Predictions': y3_pred.ravel(),
    'Actuals':          y3_true.ravel()
})
results3.to_csv('test_results_S001_3_month.csv', index=False)




3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
